# Lesson 04 - Designmønster for Verktøybruk

I denne leksjonen vil du lære designmønsteret **Verktøybruk** for AI-agenter ved bruk av Microsoft Agent Framework (Python). Vi dekker:

- Definere funksjonsverktøy med `@tool` dekoratøren og typede parametere  
- Tilby verktøyskjemaer slik at modellen vet hva hvert verktøy gjør  
- Kontrollere verktøykjøring med `approval_mode`  
- Returnere **strukturert output** via Pydantic-modeller og `response_format`  

Scenariet er en **reisebestillingsagent** som kan søke etter reisemål, sjekke tilgjengelighet og hente flyinformasjon.


## Oppsett


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Definere verktøy med @tool-dekoratøren

`@tool`-dekoratøren gjør en vanlig Python-funksjon om til et verktøy som en agent kan kalle på.
Viktige punkter:

- **Docstringen** blir verktøybeskrivelsen som modellen ser.
- **Typeannotasjoner** (inkludert `Annotated` med beskrivelser) definerer verktøyskjemaet.
- `approval_mode` styrer om brukeren må godkjenne hvert kall før det utføres.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## Lage en agent med flere verktøy

Gi alle de tre verktøyene til klienten slik at modellen kan bruke hvilke som helst den trenger for å svare på brukerens spørsmål.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = await provider.create_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## Strukturert output med verktøy

Ved å sette `response_format` til en Pydantic-modell, tvinges agenten til å returnere et veldesignet JSON-objekt i stedet for fritekst. Dette er nyttig når etterfølgende kode trenger å behandle resultatet programmessig.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = await provider.create_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## Verktøygodkjenningsmønstre

Parameteren `approval_mode` på `@tool` styrer om verktøykall krever menneskelig godkjenning før de utføres:

| Mode | Adferd |
|---|---|
| `"never_require"` | Verktøyet kjører automatisk — ingen brukergodkjenning nødvendig. |
| `"always_require"` | Hvert kall må godkjennes av brukeren før det utføres. |

Bruk `"always_require"` for verktøy som har bivirkninger (f.eks. å bestille en flyreise, belaste et kredittkort) slik at et menneske er med i prosessen.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## Sammendrag

I denne leksjonen lærte du hvordan du:

1. **Definerer verktøy** ved å bruke `@tool`-dekoren med typede parametere og dokumentasjonsstrenger som fungerer som verktøyskjema.
2. **Setter sammen flere verktøy** slik at agenten kan kalle dem i rekkefølge for å svare på komplekse spørsmål.
3. **Returnerer strukturert utdata** ved å sende en Pydantic-modell som `response_format`.
4. **Kontrollerer verktøy-godkjenning** med `approval_mode` for å holde et menneske involvert ved sensitive operasjoner.

Disse mønstrene danner grunnlaget for å bygge pålitelige, produksjonsklare agenter som kan samhandle trygt med eksterne systemer.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfraskrivelse**:
Dette dokumentet er oversatt ved hjelp av AI-oversettelsestjenesten [Co-op Translator](https://github.com/Azure/co-op-translator). Selv om vi streber etter nøyaktighet, vennligst vær oppmerksom på at automatiske oversettelser kan inneholde feil eller unøyaktigheter. Det opprinnelige dokumentet på originalspråket skal anses som den autoritative kilden. For kritisk informasjon anbefales profesjonell menneskelig oversettelse. Vi er ikke ansvarlige for eventuelle misforståelser eller feiltolkninger som oppstår ved bruk av denne oversettelsen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
